In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
from imblearn.over_sampling import SMOTE
import joblib
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully")
print("pandas:", pd.__version__)
print("numpy:", np.__version__)



output
All libraries imported successfully
pandas: 2.3.3
numpy: 2.4.6

In [ ]:
df = pd.read_excel('/kaggle/input/datasets/ankitverma2010/ecommerce-customer-churn-analysis-and-prediction/E Commerce Dataset.xlsx', sheet_name='E Comm')

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

output
Shape: (5630, 20)
Columns: ['CustomerID', 'Churn', 'Tenure', 'PreferredLoginDevice', 'CityTier', 'WarehouseToHome', 'PreferredPaymentMode', 'Gender', 'HourSpendOnApp', 'NumberOfDeviceRegistered', 'PreferedOrderCat', 'SatisfactionScore', 'MaritalStatus', 'NumberOfAddress', 'Complain', 'OrderAmountHikeFromlastYear', 'CouponUsed', 'OrderCount', 'DaySinceLastOrder', 'CashbackAmount']

In [ ]:
print("First 5 rows:")
print(df.head())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

output
First 5 rows:
   CustomerID  Churn  Tenure PreferredLoginDevice  CityTier  WarehouseToHome  \
0       50001      1     4.0         Mobile Phone         3              6.0   
1       50002      1     NaN                Phone         1              8.0   
2       50003      1     NaN                Phone         1             30.0   
3       50004      1     0.0                Phone         3             15.0   
4       50005      1     0.0                Phone         1             12.0   

  PreferredPaymentMode  Gender  HourSpendOnApp  NumberOfDeviceRegistered  \
0           Debit Card  Female             3.0                         3   
1                  UPI    Male             3.0                         4   
2           Debit Card    Male             2.0                         4   
3           Debit Card    Male             2.0                         4   
4                   CC    Male             NaN                         3   

     PreferedOrderCat  SatisfactionScore MaritalStatus  NumberOfAddress  \
0  Laptop & Accessory                  2        Single                9   
1              Mobile                  3        Single                7   
2              Mobile                  3        Single                6   
3  Laptop & Accessory                  5        Single                8   
4              Mobile                  5        Single                3   

   Complain  OrderAmountHikeFromlastYear  CouponUsed  OrderCount  \
0         1                         11.0         1.0         1.0   
1         1                         15.0         0.0         1.0   
2         1                         14.0         0.0         1.0   
3         0                         23.0         0.0         1.0   
4         0                         11.0         1.0         1.0   

   DaySinceLastOrder  CashbackAmount  
0                5.0          159.93  
1                0.0          120.90  
2                3.0          120.28  
3                3.0          134.07  
4                3.0          129.60  

Data types:
CustomerID                       int64
Churn                            int64
Tenure                         float64
PreferredLoginDevice            object
CityTier                         int64
WarehouseToHome                float64
PreferredPaymentMode            object
Gender                          object
HourSpendOnApp                 float64
NumberOfDeviceRegistered         int64
PreferedOrderCat                object
SatisfactionScore                int64
MaritalStatus                   object
NumberOfAddress                  int64
Complain                         int64
OrderAmountHikeFromlastYear    float64
CouponUsed                     float64
OrderCount                     float64
DaySinceLastOrder              float64
CashbackAmount                 float64
dtype: object

Missing values:
CustomerID                       0
Churn                            0
Tenure                         264
PreferredLoginDevice             0
CityTier                         0
WarehouseToHome                251
PreferredPaymentMode             0
Gender                           0
HourSpendOnApp                 255
NumberOfDeviceRegistered         0
PreferedOrderCat                 0
SatisfactionScore                0
MaritalStatus                    0
NumberOfAddress                  0
Complain                         0
OrderAmountHikeFromlastYear    265
CouponUsed                     256
OrderCount                     258
DaySinceLastOrder              307
CashbackAmount                   0
dtype: int64

In [ ]:
print("Churn distribution:")
print(df['Churn'].value_counts())
print("\nChurn percentage:")
print(df['Churn'].value_counts(normalize=True) * 100)

output
Churn distribution:
Churn
0    4682
1     948
Name: count, dtype: int64

Churn percentage:
Churn
0    83.161634
1    16.838366
Name: proportion, dtype: float64

In [ ]:
num_cols = ['Tenure', 'WarehouseToHome', 'HourSpendOnApp', 
            'OrderAmountHikeFromlastYear', 'CouponUsed', 
            'OrderCount', 'DaySinceLastOrder']

for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

print("Missing values after filling:")
print(df.isnull().sum())

output
Missing values after filling:
CustomerID                     0
Churn                          0
Tenure                         0
PreferredLoginDevice           0
CityTier                       0
WarehouseToHome                0
PreferredPaymentMode           0
Gender                         0
HourSpendOnApp                 0
NumberOfDeviceRegistered       0
PreferedOrderCat               0
SatisfactionScore              0
MaritalStatus                  0
NumberOfAddress                0
Complain                       0
OrderAmountHikeFromlastYear    0
CouponUsed                     0
OrderCount                     0
DaySinceLastOrder              0
CashbackAmount                 0
dtype: int64

In [ ]:
cat_cols = ['PreferredLoginDevice', 'PreferredPaymentMode', 
            'Gender', 'PreferedOrderCat', 'MaritalStatus']

le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

print("Categorical columns encoded. Sample check:")
print(df[cat_cols].head())

output
Categorical columns encoded. Sample check:
   PreferredLoginDevice  PreferredPaymentMode  Gender  PreferedOrderCat  \
0                     1                     4       0                 2   
1                     2                     6       1                 3   
2                     2                     4       1                 3   
3                     2                     4       1                 2   
4                     2                     0       1                 3   

   MaritalStatus  
0              2  
1              2  
2              2  
3              2  
4              2  

In [ ]:
X = df.drop(columns=['CustomerID', 'Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", X_train.shape, y_train.value_counts().to_dict())
print("After SMOTE:", X_train_sm.shape, y_train_sm.value_counts().to_dict())
output
Before SMOTE: (4504, 18) {0: 3741, 1: 763}
After SMOTE: (7482, 18) {0: 3741, 1: 3741}

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss'),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

results = {}

for name, model in models.items():
    model.fit(X_train_sm, y_train_sm)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    results[name] = {
        'Accuracy': round(accuracy_score(y_test, y_pred) * 100, 2),
        'Precision': round(precision_score(y_test, y_pred) * 100, 2),
        'Recall': round(recall_score(y_test, y_pred) * 100, 2),
        'F1 Score': round(f1_score(y_test, y_pred) * 100, 2),
        'ROC-AUC': round(roc_auc_score(y_test, y_prob) * 100, 2)
    }
    print(f"{name} done")

output
Logistic Regression done
Random Forest done
XGBoost done
Gradient Boosting done

In [ ]:
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('ROC-AUC', ascending=False)

print("Model Comparison:")
print(results_df.to_string())

print("\nBest model:", results_df['ROC-AUC'].idxmax())

output
Model Comparison:
                     Accuracy  Precision  Recall  F1 Score  ROC-AUC
XGBoost                 97.34      94.29   89.19     91.67    98.34
Random Forest           96.54      91.48   87.03     89.20    98.19
Gradient Boosting       89.88      67.66   73.51     70.47    92.66
Logistic Regression     74.51      36.86   77.30     49.91    84.75

Best model: XGBoost

In [ ]:
best_model = models['XGBoost']

joblib.dump(best_model, '/kaggle/working/best_model.pkl')

feature_names = X.columns.tolist()
joblib.dump(feature_names, '/kaggle/working/feature_names.pkl')

print("Model saved successfully")
print("Features saved:", feature_names)

output
Model saved successfully
Features saved: ['Tenure', 'PreferredLoginDevice', 'CityTier', 'WarehouseToHome', 'PreferredPaymentMode', 'Gender', 'HourSpendOnApp', 'NumberOfDeviceRegistered', 'PreferedOrderCat', 'SatisfactionScore', 'MaritalStatus', 'NumberOfAddress', 'Complain', 'OrderAmountHikeFromlastYear', 'CouponUsed', 'OrderCount', 'DaySinceLastOrder', 'CashbackAmount']

In [ ]:
df.to_csv('/kaggle/working/ecommerce_clean.csv', index=False)

print("Dataset saved successfully")
print("Shape:", df.shape)
print("Sample CustomerIDs:", df['CustomerID'].head(10).tolist())

output
loaded_model = joblib.load('/kaggle/working/best_model.pkl')
loaded_features = joblib.load('/kaggle/working/feature_names.pkl')
loaded_df = pd.read_csv('/kaggle/working/ecommerce_clean.csv')

test_customer = loaded_df[loaded_df['CustomerID'] == 50001][loaded_features]
prediction = loaded_model.predict(test_customer)[0]
confidence = loaded_model.predict_proba(test_customer)[0][1] * 100

print("Model loaded:", loaded_model)
print("Features loaded:", loaded_features)
print("Dataset loaded shape:", loaded_df.shape)
print("Test prediction for CustomerID 50001:")
print("Churn prediction:", prediction)
print("Churn probability:", round(confidence, 2), "%")

output
Model loaded: XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...)
Features loaded: ['Tenure', 'PreferredLoginDevice', 'CityTier', 'WarehouseToHome', 'PreferredPaymentMode', 'Gender', 'HourSpendOnApp', 'NumberOfDeviceRegistered', 'PreferedOrderCat', 'SatisfactionScore', 'MaritalStatus', 'NumberOfAddress', 'Complain', 'OrderAmountHikeFromlastYear', 'CouponUsed', 'OrderCount', 'DaySinceLastOrder', 'CashbackAmount']
Dataset loaded shape: (5630, 20)
Test prediction for CustomerID 50001:
Churn prediction: 1
Churn probability: 87.28 %